In [91]:
import nest_asyncio
import re
from crawl4ai import AsyncWebCrawler, BrowserConfig, CrawlerRunConfig, CacheMode
from langchain_core.documents import Document

# This is the magic line for Jupyter notebooks
nest_asyncio.apply()

urls = [
    "https://www.bio-monitoring.ca",
    "https://www.bio-monitoring.ca/scientific-technical-innovation",
    "https://www.bio-monitoring.ca/about-us-1",
    "https://www.bio-monitoring.ca/mission-vision",
    "https://www.bio-monitoring.ca/contact",
    "https://www.bio-monitoring.ca/product/21308262/folic-acid-supplement"
]

async def load_web_docs_parallel(urls):
    browser_config = BrowserConfig(headless=True, verbose=False)
    run_config = CrawlerRunConfig(
        cache_mode=CacheMode.BYPASS,
        semaphore_count=5,
        excluded_tags=['nav'], # <nav>
        excluded_selector='.nav, .menu, [class^="menu"], [class^="nav"]', #<div class="menu">, starts with nav or starts with menu
        remove_overlay_elements=True
    )

    documents = []

    async with AsyncWebCrawler(config=browser_config) as crawler:
        # Use arun_many for parallel execution
        results = await crawler.arun_many(urls=urls, config=run_config)

        for result in results:
            if result.success:
                # Get the raw markdown
                content = result.markdown.raw_markdown 

                doc = Document(
                    page_content=content,
                    metadata={"source": result.url}
                )
                documents.append(doc)
            else:
                print(f"Error at {result.url}: {result.error_message}")

    return documents

# In Jupyter, you can use 'await' directly or call it like this:
docs = await load_web_docs_parallel(urls)

# View the first document's content
print(f"Loaded {len(docs)} documents.")

[FETCH]... ↓ https://www.bio-monitoring.ca                                                                        |
✓ | ⏱: 3.43s 

[SCRAPE].. ◆ https://www.bio-monitoring.ca                                                                        |
✓ | ⏱: 0.01s 

[COMPLETE] ● https://www.bio-monitoring.ca                                                                        |
✓ | ⏱: 3.45s 

[FETCH]... ↓ https://www.bio-monitoring.ca/mission-vision                                                         |
✓ | ⏱: 1.55s 

[SCRAPE].. ◆ https://www.bio-monitoring.ca/mission-vision                                                         |
✓ | ⏱: 0.01s 

...

[COMPLETE] ● https://www.bio-monitoring.ca/scientific-technical-innovation                                        |
✓ | ⏱: 1.77s 

[FETCH]... ↓ https://www.bio-monitoring.ca/contact                                                                |
✓ | ⏱: 2.10s 

[SCRAPE].. ◆ https://www.bio-monitoring.ca/contact                                                                |
✓ | ⏱: 0.01s 

[COMPLETE] ● https://www.bio-monitoring.ca/contact                                                                |
✓ | ⏱: 2.11s 

Loaded 6 documents.


In [98]:
print(docs[0].page_content)#['page_content']


Cookie Policy
![bio-monitoring.ca](https://primary.jwwb.nl/public/x/q/v/temp-gaftuxzhkeplfbckxcbc/image-high-qud4yf.png?enable-io=true&enable=upscale&height=70)
bio-monitoring.ca
[ 0 ](https://www.bio-monitoring.ca/cart)
Sold out
###  [Folic Acid supplement](https://www.bio-monitoring.ca/product/21308262/folic-acid-supplement)
CA$54.25
Sold out
###  [Folic Acid supplement](https://www.bio-monitoring.ca/product/21308262/folic-acid-supplement)
CA$54.25
Natural, Active, and Organic Biomonitoring MTHFR‑Friendly Formula.
“Track your folate levels with your healthcare provider.”   

1,000 mcg / 1 mg with “Pre‑conception & Prenatal Support”   

Uses:  
“Helps prevent folate deficiency.”  
“Helps prevent neural tube defects when taken daily at least 12 weeks before and during early pregnancy, as part of a healthy diet.”  
Directions: “Adults: Take 1 tablet daily or as directed by a healthcare practitioner.”  
Cautions: Standard NHP warnings (e.g., keep out of reach of children.  

[+ See detai

In [100]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Chunk the text (VERY important for RAG)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)

chunks = text_splitter.split_documents(docs)

print(f"Loaded {len(chunks)} chunks")

Loaded 15 chunks


In [103]:
import os
from langchain_community.vectorstores import UpstashVectorStore

with open('../upstash.txt', 'r') as f:
    lines = f.readlines()
# Set your credentials
os.environ["UPSTASH_VECTOR_REST_URL"] = lines[0].strip()
os.environ["UPSTASH_VECTOR_REST_TOKEN"] = lines[1].strip()
name_space = 'bio-monitoring.ca'
# Path A: Using Upstash Hosted Embeddings (The "Better" way)
# We pass embedding=True so LangChain knows Upstash handles the vectorization
vectorstore = UpstashVectorStore(
    embedding=True, # Whether the embedding should be calculated in cloud
    namespace = name_space # Your first namespace
)

# Add your existing 'chunks' from your previous code
vectorstore.add_documents(chunks)

print(f"Docs added to namespace: {name_space}")

Docs added to namespace: bio-monitoring.ca


In [104]:
#Sanity check
query = "What is their innovation?"
results = vectorstore.similarity_search(query, k=2)

for i, doc in enumerate(results, 1):
    clean_string = " ".join(doc.page_content.split())
    print(f"\nResult {i}:\n{clean_string}...")



Result 1:
### -Producing natural, research-driven alternatives to synthetic supplements. ### -Advancing sustainable food biotechnology within Alberta’s bioeconomy. # VISION ## To become Alberta’s leading innovator in bio-based nutrition, creating a future to support human health, and reduce NTD births. **We produce highly bioactive, natural folic acid through precision fermentation, giving food, supplement, and pharma brands a cleaner, more effective folate ingredient that’s easier to absorb, more sustainable to manufacture, and fully aligned with modern regulatory and consumer expectations.** ### Chat Assistant Clear Chat...

Result 2:
## Our technology targets **clinically complex, high‑value segments** where standard synthetic folic acid may be suboptimal ## Our flagship initiative, the BioFolate Project, focuses on producing natural folic acid (vitamin B9) through fermentation using optimized _Lactobacillus_ strains. This approach reduces dependency on synthetic vitamins and suppo